# 04 — Meta-Learning v1: Three-Agent Ensemble Supervisor

Loads pre-computed OOS probabilities from **LGBM**, **Mamba**, and **TCN**, then trains
a LightGBM meta-classifier that predicts whether an ensemble signal will hit its
take-profit before stop-loss (Triple Barrier Method label).

**Prerequisites — run these first:**
1. `00_data_ingestion_v1.ipynb` — creates `BTCUSDT_1h_unified.parquet`
2. `01_lgbm_v1.ipynb` — creates `artifacts/notebooks_v2/01_lgbm/oos_probs.npy`
3. `02_mamba_v1.ipynb` (Colab) — copy output into `artifacts/notebooks_v2/02_mamba/`
4. `03_tcn_v1.ipynb` — creates `artifacts/notebooks_v2/03_tcn/oos_probs.npy`

**Architecture:**
- Ensemble P(Up) = mean of LGBM, Mamba, TCN probabilities.
- Meta-features: individual model probs + regime context from unified parquet.
- Meta-target: TBM label at the ensemble signal bar.
- Walk-forward: expanding window, 3-month step, 48h embargo.
- Sizing: position = side × (meta_prob − 0.5) × 2 when meta_prob > threshold.


In [ ]:

import calendar, json, time, warnings
from pathlib import Path

import lightgbm as lgb
import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
mpl.rcParams.update({
    'font.family':'serif','font.serif':['DejaVu Serif'],
    'axes.spines.top':False,'axes.spines.right':False,
    'axes.labelsize':10,'axes.titlesize':11,'xtick.labelsize':9,'ytick.labelsize':9,
    'legend.fontsize':9,'figure.dpi':120,'savefig.dpi':200,'savefig.bbox':'tight',
})
ACCENT='#F7931A'; BLUE='#2962FF'; GREY='#9E9E9E'; RED='#EF5350'; GREEN='#26A69A'; TEAL='#00ACC1'

def _repo_root():
    p=Path.cwd()
    while p!=p.parent:
        if (p/'pyproject.toml').exists(): return p
        p=p.parent
    raise RuntimeError('pyproject.toml not found')

REPO     = _repo_root()
A2       = REPO/'artifacts'/'notebooks_v2'
LGBM_DIR = A2/'01_lgbm'
MAMBA_DIR= A2/'02_mamba'
TCN_DIR  = A2/'03_tcn'
ARTS_DIR = A2/'04_meta'
ARTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Time splits ───────────────────────────────────────────────────────────────
META_TRAIN_START = pd.Timestamp('2022-01-01')  # earliest bar with all model probs
OOS_START        = pd.Timestamp('2024-01-01')  # meta-model evaluated from here
STEP_MONTHS      = 3
EMBARGO_H        = 48
META_THRESHOLD   = 0.55   # meta_prob > threshold → enter trade
META_TRAIN_WINDOW= 1000   # bars of history used per fold (expanding, so this is minimum)

# ── TBM params (for meta-target labels) ──────────────────────────────────────
SL_MULT=1.5; TP_MULT=2.5; MAX_HOLD=48

# ── Ensemble signal thresholds ────────────────────────────────────────────────
ENS_LONG_THR  = 0.56
ENS_SHORT_THR = 0.44

# ── Fee model ─────────────────────────────────────────────────────────────────
TAKER_FEE=0.0005; FUNDING_H=0.0000077

# ── Meta-features: model probs + regime context ───────────────────────────────
REGIME_FEATURES = [
    'atr_14_pct', 'hurst_24h', 'hurst_168h', 'vol_ratio_24h',
    'bb_width_pct', 'rsi_14', 'ret_1h', 'ret_24h',
    'sideways_flag', 'tfi_z_24h', 'hour_sin', 'hour_cos',
    'halving_cycle_pos', 'trend_score',
]
print(f'Artifacts → {ARTS_DIR}')


## 1 · Load unified parquet

In [ ]:

UNIFIED=REPO/'data'/'features'/'BTCUSDT_1h_unified.parquet'
if not UNIFIED.exists():
    raise FileNotFoundError(f'{UNIFIED} — run 00_data_ingestion_v1.ipynb first.')

df=pd.read_parquet(UNIFIED)
df.index=df.index.tz_localize(None) if df.index.tz else df.index
print(f'Unified: {df.shape}  ({df.index.min().date()} → {df.index.max().date()})')
avail=[f for f in REGIME_FEATURES if f in df.columns]
miss =[f for f in REGIME_FEATURES if f not in df.columns]
if miss: print(f'WARNING — missing regime features: {miss}')


## 2 · Load base model OOS probabilities

In [ ]:

def _load_model_probs(art_dir, name):
    probs_path = art_dir / 'oos_probs.npy'
    index_path = art_dir / 'oos_index.npy'
    if not probs_path.exists():
        print(f'WARNING — {name} probs not found at {probs_path}')
        return None
    probs = np.load(probs_path)
    idx   = pd.to_datetime(np.load(index_path))
    s     = pd.Series(probs, index=idx, name=f'{name}_p_up')
    print(f'  {name:<8}: {len(s):,} bars  ({s.index.min().date()} → {s.index.max().date()})'
          f'  mean={s.mean():.3f}')
    return s

print('Loading model OOS probabilities:')
lgbm_p   = _load_model_probs(LGBM_DIR,  'lgbm')
mamba_p  = _load_model_probs(MAMBA_DIR, 'mamba')
tcn_p    = _load_model_probs(TCN_DIR,   'tcn')

available = {k:v for k,v in {'lgbm':lgbm_p,'mamba':mamba_p,'tcn':tcn_p}.items() if v is not None}
if len(available) < 2:
    raise RuntimeError(f'Need at least 2 models, got {list(available)}. Run model notebooks first.')
print(f'\nAvailable models: {list(available)}')


## 3 · Align to common index & build signal DataFrame

In [ ]:

# Align all series to the unified parquet index
common_idx = df.index
for name, s in available.items():
    df[f'{name}_p_up'] = s.reindex(common_idx)

model_cols = [f'{m}_p_up' for m in available]
df['ensemble_p_up'] = df[model_cols].mean(axis=1)

# Ensemble signal: fire when ensemble crosses threshold
df['signal_long']  = (df['ensemble_p_up'] > ENS_LONG_THR).astype(int)
df['signal_short'] = (df['ensemble_p_up'] < ENS_SHORT_THR).astype(int)
df['primary_signal'] = df['signal_long'] | df['signal_short']
df['primary_side']   = df['signal_long'] * 1 + df['signal_short'] * (-1)

signal_bars = df[df['primary_signal']==1]
n_long  = (signal_bars['primary_side']==1).sum()
n_short = (signal_bars['primary_side']==-1).sum()
print(f'Signal bars: {len(signal_bars):,}  (Long={n_long:,}  Short={n_short:,})')
print(f'Signal rate: {len(signal_bars)/len(df)*100:.1f}%')

# Correlation table
print('\nModel probability correlations (Spearman):')
print(df[model_cols].corr(method='spearman').round(3).to_string())


## 4 · Compute TBM labels for meta-target

In [ ]:

close_arr = df['close'].values
atr_arr   = df['atr_14_pct'].values
n         = len(df)
meta_y    = np.full(n, np.nan)

print(f'Computing TBM labels (SL={SL_MULT}×ATR  TP={TP_MULT}×ATR  max_hold={MAX_HOLD}h) ...')
t0=time.time()
for i in range(n - MAX_HOLD):
    if not df['primary_signal'].iloc[i]: continue
    if np.isnan(atr_arr[i]) or atr_arr[i]==0: continue
    side   = df['primary_side'].iloc[i]
    tp = close_arr[i] * (1 + side * TP_MULT * atr_arr[i])
    sl = close_arr[i] * (1 - side * SL_MULT * atr_arr[i])
    for j in range(i+1, min(i+MAX_HOLD+1, n)):
        if side==1:
            if close_arr[j]>=tp: meta_y[i]=1; break
            if close_arr[j]<=sl: meta_y[i]=0; break
        else:
            if close_arr[j]<=tp: meta_y[i]=1; break
            if close_arr[j]>=sl: meta_y[i]=0; break

df['meta_y'] = meta_y
valid = df['meta_y'].notna() & df['primary_signal'].astype(bool)
print(f'Done in {time.time()-t0:.0f}s  |  valid signal bars: {valid.sum():,}  '
      f'positive rate={df.loc[valid,"meta_y"].mean():.3f}')


## 5 · Walk-forward meta-model training

In [ ]:

META_FEATURES = model_cols + ['ensemble_p_up'] + [f for f in avail if f in df.columns]

# Keep only bars where primary_signal fired and meta_y is valid
sig_df = df[valid].copy()

# WFO loop: expanding window, 3-month steps
step_dates = []
d = META_TRAIN_START + pd.DateOffset(months=3)
while d <= df.index[-1]:
    step_dates.append(d); d += pd.DateOffset(months=STEP_MONTHS)

oos_meta_probs = pd.Series(np.nan, index=sig_df.index, name='meta_prob')
fold_log = []; last_fi = None

for fi, step_end in enumerate(step_dates):
    if step_end <= META_TRAIN_START: continue
    step_start = step_end
    step_stop  = step_end + pd.DateOffset(months=STEP_MONTHS)

    # Training data: everything before step_end minus embargo
    purge_cut   = step_start - pd.Timedelta(hours=EMBARGO_H)
    train_mask  = sig_df.index < purge_cut
    if train_mask.sum() < META_TRAIN_WINDOW // 2: continue

    X_tr = sig_df.loc[train_mask, META_FEATURES].fillna(0).values
    y_tr = sig_df.loc[train_mask, 'meta_y'].values.astype(int)
    if len(np.unique(y_tr)) < 2: continue

    mdl = lgb.LGBMClassifier(
        n_estimators=200, num_leaves=15, max_depth=4, learning_rate=0.05,
        subsample=0.7, colsample_bytree=0.7, reg_alpha=0.1, reg_lambda=1.0,
        min_child_samples=20, objective='binary', metric='auc',
        verbose=-1, random_state=42,
    )
    mdl.fit(X_tr, y_tr)

    # Predict OOS block
    oos_mask = (sig_df.index >= step_start) & (sig_df.index < step_stop)
    if oos_mask.sum() == 0: continue
    X_oos = sig_df.loc[oos_mask, META_FEATURES].fillna(0).values
    oos_meta_probs[sig_df.index[oos_mask]] = mdl.predict_proba(X_oos)[:, 1]
    fold_log.append({'fold':fi,'train_bars':int(train_mask.sum()),
                     'oos':f'{step_start.date()}→{step_stop.date()}','oos_bars':int(oos_mask.sum())})
    last_fi = mdl
    if fi % 4 == 0:
        print(f'  fold {fi}: train<{purge_cut.date()}  OOS {step_start.date()}→{step_stop.date()}  '
              f'({int(oos_mask.sum())} signal bars)')

print(f'\nMeta WFO done  |  {len(fold_log)} folds  |  OOS meta probs: {oos_meta_probs.notna().sum():,}')

# Feature importance
if last_fi:
    fi_dict = dict(zip(META_FEATURES, last_fi.feature_importances_))
    fi_sorted = sorted(fi_dict.items(), key=lambda x: x[1], reverse=True)
    print('\nFeature importances (last fold):')
    for name, imp in fi_sorted[:15]: print(f'  {name:<30s}  {imp:>6.0f}')


## 6 · Meta-model evaluation

In [ ]:

oos_sig = sig_df[sig_df.index >= OOS_START].copy()
oos_sig['meta_prob'] = oos_meta_probs.reindex(oos_sig.index)
valid_oos = oos_sig['meta_prob'].notna() & oos_sig['meta_y'].notna()

if valid_oos.sum() > 0:
    auc_meta = roc_auc_score(oos_sig.loc[valid_oos,'meta_y'], oos_sig.loc[valid_oos,'meta_prob'])
    print(f'Meta-model OOS AUC: {auc_meta:.4f}  (signal bars: {valid_oos.sum():,})')
    print(f'Approval rate (meta_prob > {META_THRESHOLD}): '
          f'{(oos_sig.loc[valid_oos,"meta_prob"] > META_THRESHOLD).mean():.1%}')
    print(f'Primary signal rate vs approved: '
          f'{len(oos_sig):,} signals → '
          f'{(oos_sig.loc[valid_oos,"meta_prob"] > META_THRESHOLD).sum():,} approved')
else:
    print('No OOS meta predictions — check that Mamba artifacts are copied.')
    auc_meta = float('nan')


## 7 · Sized backtest (meta-filtered)

In [ ]:

# Build a continuous sizing series over the OOS bars
oos_df = df[df.index >= OOS_START].copy()
sizing  = pd.Series(0.0, index=oos_df.index)

for ts, row in oos_sig[oos_sig.index >= OOS_START].iterrows():
    meta_prob = oos_meta_probs.get(ts, np.nan)
    if np.isnan(meta_prob) or meta_prob <= META_THRESHOLD: continue
    size = row['primary_side'] * (meta_prob - 0.5) * 2.0  # ∈ (-1, 1)
    sizing[ts] = size

# Simple position-hold backtest using ATR exits
close_arr = oos_df['close'].values
atr_arr   = oos_df['atr_14_pct'].values
sz_arr    = sizing.values

n = len(close_arr); eq = np.ones(n); cur = 1.0; trades = []
in_pos = False; direction = None; entry_px = sl_px = tp_px = pos_eq = entry_fee = 0.0
hold_cnt = 0; funding = 0.0; max_hold_bt = MAX_HOLD; sl_m = 1.5; tp_m = 2.5

for i in range(n):
    px = close_arr[i]
    if in_pos:
        hold_cnt += 1
        if direction=='short': funding += FUNDING_H
        eq[i] = pos_eq * (px/entry_px if direction=='long' else 1+(entry_px-px)/entry_px)
        exited=False; exit_px=0.; net=0.
        if hold_cnt >= 4:
            atr=max(atr_arr[i], 0.01)
            if direction=='long':
                if px<=sl_px: exit_px=sl_px;exited=True
                elif px>=tp_px: exit_px=tp_px;exited=True
                elif hold_cnt>=max_hold_bt: exit_px=px;exited=True
            else:
                if px>=sl_px: exit_px=sl_px;exited=True
                elif px<=tp_px: exit_px=tp_px;exited=True
                elif hold_cnt>=max_hold_bt: exit_px=px;exited=True
        if exited:
            gross=((exit_px-entry_px)/entry_px if direction=='long' else (entry_px-exit_px)/entry_px)
            net=gross-TAKER_FEE*2+funding
            cur=pos_eq*(1.+net); eq[i]=cur
            trades.append({'direction':direction,'gross':gross,'net':net,'hold':hold_cnt})
            in_pos=False; funding=0.
    elif sz_arr[i]!=0 and not in_pos:
        direction='long' if sz_arr[i]>0 else 'short'
        atr=max(atr_arr[i],0.01); entry_px=px; pos_eq=cur; hold_cnt=0; funding=0.
        entry_fee=TAKER_FEE
        if direction=='long': sl_px=px*(1-sl_m*atr); tp_px=px*(1+tp_m*atr)
        else: sl_px=px*(1+sl_m*atr); tp_px=px*(1-tp_m*atr)
        in_pos=True; eq[i]=cur
    else: eq[i]=cur

TF_meta=pd.DataFrame(trades) if trades else pd.DataFrame(columns=['direction','gross','net','hold'])
def _sharpe(e): r=np.diff(np.log(np.maximum(e,1e-12))); return float(r.mean()/(r.std(ddof=1)+1e-12)*np.sqrt(24*365))
def _maxdd(e):  pk=np.maximum.accumulate(e); return float(((e-pk)/(pk+1e-12)).min())

bh=(close_arr/close_arr[0]-1)*100
wr=float((TF_meta['net']>0).mean()) if len(TF_meta) else 0.
print(f'Meta-agent OOS: Return={eq[-1]-1:+.2%}  Sharpe={_sharpe(eq):+.3f}  MaxDD={_maxdd(eq):.2%}  '
      f'Trades={len(TF_meta)}  WinRate={wr:.1%}')


## 8 · Comparison equity curves

In [ ]:

o_idx = oos_df.index
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), height_ratios=[3,1], sharex=True)

def _load_eq(art_dir, best_params_key='backtest_wfees'):
    rj = art_dir / 'results.json'
    if not rj.exists(): return None, None
    r = json.load(open(rj))
    oos_p = np.load(art_dir/'oos_probs.npy')
    oos_i = pd.to_datetime(np.load(art_dir/'oos_index.npy'))
    return pd.Series(oos_p, index=oos_i), r

lgbm_s, lgbm_r = _load_eq(LGBM_DIR)
mamba_s, mamba_r = _load_eq(MAMBA_DIR)
tcn_s,  tcn_r  = _load_eq(TCN_DIR)

colors_map={'lgbm':ACCENT,'mamba':BLUE,'tcn':TEAL,'meta':GREEN}
for name,(series,results) in [('lgbm',(lgbm_s,lgbm_r)),('mamba',(mamba_s,mamba_r)),('tcn',(tcn_s,tcn_r))]:
    if series is None: continue
    bp=results.get('best_params',{})
    long_thr=bp.get('long_threshold',0.55); short_thr=bp.get('short_threshold',0.35)
    p=series.reindex(o_idx)
    eq_=np.ones(len(o_idx)); cur_=1.0
    for i,v in enumerate(p.values):
        if not np.isnan(v):
            if v>long_thr: cur_=cur_*1.001  # simplified for comparison overlay
            elif v<short_thr: cur_=cur_*0.999
        eq_[i]=cur_
    ret=(series.reindex(o_idx).dropna())
    sharpe=results.get('backtest_wfees',{}).get('sharpe','n/a')
    total_ret=results.get('backtest_wfees',{}).get('total_ret','n/a')
    ax1.plot(o_idx,(np.ones(len(o_idx))*np.nan),
             color=colors_map[name],lw=1.3,label=f'{name.upper()} Sharpe={sharpe} Ret={total_ret}')

# Meta equity
ax1.plot(o_idx,(eq-1)*100, color=GREEN, lw=2.0,
         label=f'Meta Sharpe={_sharpe(eq):+.2f} Ret={eq[-1]-1:+.1%}',zorder=5)
ax1.plot(o_idx, bh, color=GREY, lw=1.0, ls=':', label='BTC B&H')
ax1.axhline(0,color=GREY,lw=0.6,ls=':'); ax1.set_ylabel('Return (%)'); ax1.legend(fontsize=8)
ax1.set_title(f'Ensemble Meta-Agent vs Base Models — OOS {OOS_START.date()}→{o_idx[-1].date()}',fontweight='bold')
pk=np.maximum.accumulate(eq); dd=(eq-pk)/pk*100
ax2.fill_between(o_idx,dd,0,color=RED,alpha=0.4); ax2.set_ylabel('DD (%)')
ax2.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,4,7,10]))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
plt.setp(ax2.xaxis.get_majorticklabels(),rotation=30,ha='right')
fig.tight_layout(); fig.savefig(ARTS_DIR/'01_equity_comparison.png'); plt.show()


In [ ]:

# Monthly returns
eqs=pd.Series(eq,index=o_idx); mret=eqs.resample('ME').last().pct_change().fillna(0)*100
fig,ax=plt.subplots(figsize=(13,4))
ax.bar(mret.index,mret.values,color=[GREEN if r>=0 else RED for r in mret],width=22,alpha=0.75)
ax.plot(mret.index,mret.rolling(3,min_periods=1).mean(),color=BLUE,lw=2,label='3-mo SMA')
ax.axhline(0,color=GREY,lw=0.8,ls=':'); ax.set_ylabel('Return (%)'); ax.legend()
ax.set_title(f'Monthly Returns — Meta-Agent | positive {int((mret>0).sum())}/{len(mret)} avg {mret.mean():+.2f}%',fontweight='bold')
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,4,7,10]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
plt.setp(ax.xaxis.get_majorticklabels(),rotation=30,ha='right')
fig.tight_layout(); fig.savefig(ARTS_DIR/'02_monthly_returns.png'); plt.show()

# Feature importance
if last_fi and len(fi_sorted) > 0:
    top20=fi_sorted[:20]; names_=[n for n,_ in top20]; imps_=[v for _,v in top20]
    fig,ax=plt.subplots(figsize=(9,max(5,len(top20)*0.4)))
    ax.barh(range(len(top20)),imps_,color=BLUE,alpha=0.75)
    ax.set_yticks(range(len(top20))); ax.set_yticklabels(names_,fontsize=9); ax.invert_yaxis()
    ax.set_xlabel('Feature importance'); ax.set_title('Meta-model Feature Importance (last fold)',fontweight='bold')
    fig.tight_layout(); fig.savefig(ARTS_DIR/'03_feature_importance.png'); plt.show()


## 9 · Save results

In [ ]:

results={
    'notebook':'04_meta_learning_v1','created':pd.Timestamp.now().isoformat(),
    'base_models':list(available),
    'ensemble_thresholds':{'long':ENS_LONG_THR,'short':ENS_SHORT_THR},
    'meta_config':{'threshold':META_THRESHOLD,'sl_mult':SL_MULT,'tp_mult':TP_MULT,
                   'max_hold':MAX_HOLD,'step_months':STEP_MONTHS,'embargo_h':EMBARGO_H},
    'oos_period':f'{OOS_START.date()}→{o_idx[-1].date()}',
    'meta_auc_oos':round(float(auc_meta) if not np.isnan(auc_meta) else 0,4),
    'meta_agent':{
        'total_ret':round(float(eq[-1]-1),4),'sharpe':round(_sharpe(eq),4),
        'maxdd':round(_maxdd(eq),4),'n_trades':len(TF_meta),'win_rate':round(wr,4),
    },
    'wfo_folds':fold_log,
    'feature_importance':{n:int(v) for n,v in fi_sorted} if last_fi else {},
    'monthly':{'mean_pct':round(float(mret.mean()),3),'positive_months':int((mret>0).sum()),
               'total_months':int(len(mret))},
}
with open(ARTS_DIR/'results.json','w') as f: json.dump(results,f,indent=2,default=str)
print(f'Artifacts saved → {ARTS_DIR}')
print(json.dumps({k:v for k,v in results.items() if k!='wfo_folds'},indent=2)[:1500])
